In [1]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO

import pyblock
from scipy.optimize import curve_fit

from myfit import *
from myfunctions import *
from mysetup import *

from jup_plot import *

In [17]:
[x for x in codes_all]

['TurboRVB',
 'TurboRVB_DMCv1',
 'TurboRVB_DMCv2',
 'CASINO',
 'CASINO_limdmc5',
 'CASINO_limdmc6t',
 'QMCPACK',
 'QMCPACK_final',
 'CHAMP_Holland',
 'CHAMP_Cornell_Paris',
 'QMC=Chem',
 'QMeCha',
 'PyQMC',
 'Amolqc',
 'QWalk',
 'CMQMC',
 'TurboRVB_sandro',
 'QMCPACK_complex',
 'CASINO_limdmc4',
 'CASINO_limdmc5_j+',
 'CASINO_limdmc5_cbycJbufF',
 'CASINO_limdmc5t']

In [14]:
codes_all = ['TurboRVB', 'TurboRVB_DMCv1', 'TurboRVB_DMCv2', 'CASINO', 'CASINO_limdmc5', 'CASINO_limdmc6t', 'QMCPACK', 'CHAMP_Holland', 'CHAMP_Cornell_Paris', 'QMC=Chem', 'QMeCha', 'PyQMC', 'Amolqc', 'QWalk', 'CMQMC', 'TurboRVB_sandro', 'QMCPACK_complex', 'CASINO_limdmc4', 'CASINO_limdmc5_j+', 'CASINO_limdmc5_cbycJbufF', 'CASINO_limdmc5t']



colors = {
    'TurboRVB' : 'red',
    'TurboRVB_DMCv1' : 'tomato',
    'TurboRVB_DMCv2' : 'lightsalmon',
    'CASINO'   : 'green',
    'CASINO_limdmc5'   : 'green',
    'CASINO_limdmc6t'   : 'lime',
    'QMCPACK'  : 'blue',
    'QMCPACK_final'  : 'blue',
    'CHAMP_Holland' : 'blueviolet', # set here the final name for Claudia Filippi's version
    'CHAMP_Cornell_Paris' : 'magenta',    # set here the final name for Cyrus Umrigar's version
    'QMC=Chem' : 'cyan',
    'QMeCha'   : 'orange',
    'PyQMC'    : 'lightsalmon',
    'Amolqc'   : 'brown',
    'QWalk'    : 'olive',
    'CMQMC'    : 'darkgray',
}

colors_final = {
    'Amolqc': '#e6194B',
    'CASINO': '#f58231',
    'CHAMP_Cornell_Paris':'#ffe119',
    'CHAMP_Holland': '#bfef45',
    'CMQMC': '#3cb44b',
    'PyQMC': '#42d4f4',
    'QMC=Chem': '#4363d8',
    'QMCPACK_final': '#911eb4',
    'QMeCha': '#f032e6',
    'QWalk': '#800000',
    'TurboRVB': '#808000',
    'TurboRVB_DMCv2': '#469990'
}


codes = list( colors.keys() )


systems = [ 'm', 'w', 'mw', 'm---w' ]
methods = [ 'LA', 'TM', 'DLA', 'DLTM' ]

syst_names = dict(zip(systems,['methane','water','complex','far']))

colors_opt = {
    'TurboRVB_sandro'  : 'lightsalmon',
    'QMCPACK_complex'  : 'skyblue',
    'CASINO_limdmc4'   : 'gold',
    'CASINO_limdmc5_j+'    : 'mediumseagreen',
    'CASINO_limdmc5_cbycJbufF'    : 'darkgreen',
    'CASINO_limdmc5t'   : 'yellow',
    #'CASINO_pw1000'    : 'lime',
    #'CASINO_pw1000_andes' : 'mediumseagreen',
}
methods_opt = ['LA0', 'TM0']
codes_opt = list( colors_opt.keys() )
colors_all = { **colors, **colors_opt }

codes_all = codes + codes_opt

taumax = 0.1

def x_axis( df, code ):
    x = np.array( df.index )
    if code.startswith('TurboRVB') and not ('DMC' in code):
        xnew = x**2
    elif code.startswith('QMC=Chem'):
        xnew = x #* 20
    else:
        xnew = x
    return xnew

In [15]:
codes_all

['TurboRVB',
 'TurboRVB_DMCv1',
 'TurboRVB_DMCv2',
 'CASINO',
 'CASINO_limdmc5',
 'CASINO_limdmc6t',
 'QMCPACK',
 'QMCPACK_final',
 'CHAMP_Holland',
 'CHAMP_Cornell_Paris',
 'QMC=Chem',
 'QMeCha',
 'PyQMC',
 'Amolqc',
 'QWalk',
 'CMQMC',
 'TurboRVB_sandro',
 'QMCPACK_complex',
 'CASINO_limdmc4',
 'CASINO_limdmc5_j+',
 'CASINO_limdmc5_cbycJbufF',
 'CASINO_limdmc5t']

In [10]:
code_final_list = ['Amolqc', 'CASINO', 'CHAMP_Cornell_Paris', 'CHAMP_Holland', 'CMQMC', 'PyQMC', 'QMC=Chem', 'QMCPACK_final', 'QMeCha', 'QWalk', 'TurboRVB', 'TurboRVB_DMCv2']

data = {}
dataEb = {}
dataEb_fit = {} # Fitted binding energy and errors
final_Eb = {} # Final extrapolated binding energy to timestep = 0
Eb_DLA = {}
for code in codes_all:
    data[code] = {}
    dataEb[code] = {}
    dataEb_fit[code] = {}

for code in code_final_list:
    final_Eb[code] = {}


for code in codes_all:
    if code.startswith('TurboRVB') and not ('DMC' in code):
        names = ['a','ene','err']
    else:
        names = ['tau','ene','err']
    for method in methods+methods_opt:
        if 'QMCPACK' in code and method == 'DLTM':
            continue
        for syst in systems:
            key = f'{syst}_{method}'
            try: 
                data[code][key] = pd.read_csv(
                    f'../{code}/energy_{key}.csv',
                    names=names, decimal='.',
                    header=None, index_col=names[0] )
            except:
                data[code][key] = None
                #print(f'No data for {code}, {method}, {syst}')

# Compute binding energies

# Hartree to meV
c = 27.211396641308*1000

for code in codes_all:
    for method in methods:
        try:
            # Computing Binding energy for bound - far
            mw = data[code][f'mw_{method}']
            far = data[code][f'm---w_{method}']
            Eb = c*(mw.ene - far.ene)
            Eb_err = c*(far.err**2+mw.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_far_{method}'] = df_Eb.dropna()

            # Calculate the size consistency error
            m = data[code][f'm_{method}']
            w = data[code][f'w_{method}']
            far = data[code][f'm---w_{method}']
            Eb = c*(far.ene-m.ene-w.ene)
            Eb_err = c*(far.err**2+m.err**2+w.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_SCE_{method}'] = df_Eb.dropna()

            # Computing Binding energy for bound - methane - water

            m = data[code][f'm_{method}']
            w = data[code][f'w_{method}']
            mw = data[code][f'mw_{method}']
            Eb = c*(mw.ene-m.ene-w.ene)
            Eb_err = c*(mw.err**2+m.err**2+w.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_{method}'] = df_Eb.dropna()
            if method == 'DLA':
                if code in code_final_list:
                    Eb_DLA[code] = df_Eb.dropna()
                elif code == 'CASINO_limdmc5':
                    Eb_DLA['CASINO'] = df_Eb.dropna()
                elif code == 'QMCPACK' and method in ['TM','DLA']:
                    Eb_DLA['QMCPACK_final'] = df_Eb.dropna()
                elif code == 'QMCPACK_complex' and method == 'LA':
                    Eb_DLA['QMCPACK_final'] = df_Eb.dropna()


            # Fit curve to the data
            datac1 = df_Eb
            datac1['x'] = x_axis( datac1, code )
            datac = datac1[ datac1.x <= taumax ]
            datac = datac.dropna()
            x = datac.x
            y = datac.ene
            y_err = datac.err
            fun = fun_quad
            xdata = x.to_numpy()
            ydata = y.to_numpy()
            sigma = y_err.to_numpy()
            popt, pcov = curve_fit( fun, xdata=xdata, ydata=ydata, sigma=sigma )
            chi2, ndof, alpha = get_chi2_alpha_fitfun( xdata, ydata, sigma, fun, popt )

            space_pts = 101
            xlinspace = np.linspace( 0, taumax, space_pts)
            funlinspace = fun( xlinspace, *popt )

            # Fit error curve to the data
            xfit, m, s = fit_err(xdata,ydata,sigma,fitfun=fun,xmax=taumax,Nx=space_pts)


            dataEb_fit[code][f'Eb_{method}'] = {'x': xdata, 'Eb': ydata , 'Eb_err': y_err, 'x_fit': xlinspace, 'Eb_fit': funlinspace ,  'Eb_fit_lower_lim': m-1*s,  'Eb_fit_upper_lim': m+1*s}
            lim_Eb = funlinspace[0]
            lim_Eb_upper = (m+1*s)[0]
            lim_Eb_lower = (m-1*s)[0]
            lim_Eb_sigma = np.max([abs(lim_Eb - lim_Eb_upper), abs(lim_Eb - lim_Eb_lower)])

            if code in code_final_list:
                final_Eb[code][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'CASINO_limdmc5':
                final_Eb['CASINO'][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'QMCPACK' and method in ['TM','DLA']:
                final_Eb['QMCPACK_final'][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'QMCPACK_complex' and method == 'LA':
                final_Eb['QMCPACK_final'][method] = [lim_Eb, lim_Eb_sigma]
        except:
            pass

In [13]:
final_Eb

{'Amolqc': {},
 'CASINO': {},
 'CHAMP_Cornell_Paris': {},
 'CHAMP_Holland': {},
 'CMQMC': {},
 'PyQMC': {},
 'QMC=Chem': {},
 'QMCPACK_final': {},
 'QMeCha': {},
 'QWalk': {},
 'TurboRVB': {},
 'TurboRVB_DMCv2': {}}

In [6]:
for i in Eb_DLA:
    print(i)

In [50]:
Eb_DLA['CASINO'].loc[0.0050]['ene']

-29.983445880360964

In [51]:
np.save('DMC_DLA_timestep.npy',Eb_DLA, allow_pickle=True)

In [53]:
a = np.load('DMC_DLA_timestep.npy',allow_pickle=True).item()

In [37]:
np.save('final_DMC_numbers.npy',final_Eb,allow_pickle=True)

In [38]:
a = np.load('final_DMC_numbers.npy',allow_pickle=True).item()

In [39]:
a

{'CASINO': {'LA': [-32.770972861813156, 1.1217264396762694],
  'TM': [-27.222485022146845, 1.160418356152988],
  'DLA': [-31.207143284745282, 1.106168939728672],
  'DLTM': [-26.849283468095386, 0.8800110383555655]},
 'TurboRVB': {'TM': [-26.99324043703631, 2.154808496259406],
  'DLA': [-29.842584346945824, 2.095857946028719],
  'DLTM': [-28.07584279383586, 0.6199888531558564]},
 'TurboRVB_DMCv2': {'TM': [-23.222039069355723, 2.6504359063419862]},
 'QMCPACK_final': {'TM': [-24.46771599771541, 0.870303890991174],
  'DLA': [-28.903712086538437, 1.0221939564755758],
  'LA': [-17.479995393531357, 1.4655025323761421]},
 'CMQMC': {'LA': [-25.74467910935814, 1.2334255969021335],
  'TM': [-28.194552944563583, 0.7259075458747795],
  'DLA': [-27.88884282258732, 0.7616098718901441],
  'DLTM': [-27.800896955083306, 0.7943047933021923]},
 'CHAMP_Holland': {'LA': [-22.939481228433426, 0.7944575688362754],
  'TM': [-25.170845939315928, 0.5871075058334014]},
 'CHAMP_Cornell_Paris': {'LA': [-25.82287493

In [7]:
dataEb_fit

{'TurboRVB': {'Eb_TM': {'x': array([0.0025, 0.01  , 0.0225, 0.04  , 0.0625, 0.09  ]),
   'Eb': array([-28.5820619 , -25.66200693, -30.65347505, -28.81074648,
          -31.44532669, -30.23901827]),
   'Eb_err': a
   0.05    2.445802
   0.10    2.362586
   0.15    2.469022
   0.20    2.453419
   0.25    2.451748
   0.30    2.476754
   Name: err, dtype: float64,
   'x_fit': array([0.   , 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008,
          0.009, 0.01 , 0.011, 0.012, 0.013, 0.014, 0.015, 0.016, 0.017,
          0.018, 0.019, 0.02 , 0.021, 0.022, 0.023, 0.024, 0.025, 0.026,
          0.027, 0.028, 0.029, 0.03 , 0.031, 0.032, 0.033, 0.034, 0.035,
          0.036, 0.037, 0.038, 0.039, 0.04 , 0.041, 0.042, 0.043, 0.044,
          0.045, 0.046, 0.047, 0.048, 0.049, 0.05 , 0.051, 0.052, 0.053,
          0.054, 0.055, 0.056, 0.057, 0.058, 0.059, 0.06 , 0.061, 0.062,
          0.063, 0.064, 0.065, 0.066, 0.067, 0.068, 0.069, 0.07 , 0.071,
          0.072, 0.073, 0.074, 0.075, 0.076,

In [13]:
dataEb_fit['CASINO'] = dataEb_fit['CASINO_limdmc5']
dataEb_fit['QMCPACK_final'] = {'Eb_LA': dataEb_fit['QMCPACK_complex']['Eb_LA'],
                               'Eb_TM': dataEb_fit['QMCPACK']['Eb_TM'],
                               'Eb_DLA': dataEb_fit['QMCPACK']['Eb_DLA']}
# Plotting the binding energy curves for all of the codes

fig, axs = plt.subplots(2,4, figsize=(6.69,4.5), sharey=True, sharex=True,dpi=600,constrained_layout=True)

# Plot the top row binding energies
for indexj, j in enumerate(['Eb_LA','Eb_TM','Eb_DLA','Eb_DLTM']):
    for i in ['CASINO', 'TurboRVB','TurboRVB_DMCv2','QMCPACK_final','CMQMC']:
        if i == 'TurboRVB_DMCv2' and j != 'Eb_TM':
            continue

        if j == 'Eb_LA' and i == 'TurboRVB':
            continue
        
        if j == 'Eb_DLTM' and i == 'QMCPACK_final':
            continue
        print(j,i)

        code_name = i

        axs[0,indexj].errorbar(dataEb_fit[code_name][j]['x'], dataEb_fit[code_name][j]['Eb'],yerr = dataEb_fit[code_name][j]['Eb_err'], fmt='o', markersize=3,elinewidth=1, label=code_name, color=codes_color_dict[code_name])
        axs[0,indexj].plot( dataEb_fit[code_name][j]['x_fit'], dataEb_fit[code_name][j]['Eb_fit'], '--', linewidth=1, color=codes_color_dict[code_name], alpha = 0.6 )
        axs[0,indexj].fill_between(dataEb_fit[code_name][j]['x_fit'],dataEb_fit[code_name][j]['Eb_fit_lower_lim'],dataEb_fit[code_name][j]['Eb_fit_upper_lim'],color=codes_color_dict[code_name],alpha=0.2)
    axs[0,indexj].axhline(y=-27,color='black',ls='--',label='CCSD(T)')


# Plot the top row binding energies
for indexj, j in enumerate(['Eb_LA','Eb_TM','Eb_DLA']):
    for i in ['CHAMP_Holland','CHAMP_Cornell_Paris','QMC=Chem','QMeCha','PyQMC','Amolqc','QWalk']:
        if j == 'Eb_LA' and i in ['QMC=Chem','Amolqc']: #'TurboRVB':
            continue
        elif j == 'Eb_TM' and i in ['QMC=Chem','QMeCha','Amolqc','CHAMP_Holland','CHAMP_Cornell_Paris']: #'TurboRVB':
            continue
        elif j == 'Eb_DLA' and i in ['QWalk','CHAMP_Holland','CHAMP_Cornell_Paris']: #'TurboRVB':
            continue
        code_name = i
        axs[1,indexj].errorbar(dataEb_fit[code_name][j]['x'], dataEb_fit[code_name][j]['Eb'],yerr = dataEb_fit[code_name][j]['Eb_err'], fmt='o', markersize=3,elinewidth=1, label=code_name, color=codes_color_dict[code_name])
        axs[1,indexj].plot( dataEb_fit[code_name][j]['x_fit'], dataEb_fit[code_name][j]['Eb_fit'], '--', color=codes_color_dict[code_name], linewidth=1, alpha = 0.6 )
        axs[1,indexj].fill_between(dataEb_fit[code_name][j]['x_fit'],dataEb_fit[code_name][j]['Eb_fit_lower_lim'],dataEb_fit[code_name][j]['Eb_fit_upper_lim'],color=codes_color_dict[code_name],alpha=0.2)
    
    axs[1,indexj].axhline(y=-27,color='black',ls='--',label='CCSD(T)')



for i in code_final_list:
    axs[1,3].bar([1],[5],label=i, color=codes_color_dict[i])

axs[1,3].axis('off')
axs[1,3].legend(fancybox=False, fontsize=7)


axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DLTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylim([-70,-20])
axs[0,0].set_ylabel('Binding energy [meV]')
axs[1,0].set_ylabel('Binding energy [meV]')

axs[1,0].set_xticks([0.00,0.03,0.06,0.09])
axs[1,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,1].set_xticks([0.00,0.03,0.06,0.09])
axs[1,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,2].set_xticks([0.00,0.03,0.06,0.09])
axs[1,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

# axs[0,3].set_xticks([0.00,0.03,0.06,0.09])
# axs[0,3].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
# axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )


plt.savefig('Fig_Eb_fit_Ben.png')

Eb_LA CASINO
Eb_LA QMCPACK_final
Eb_LA CMQMC
Eb_TM CASINO
Eb_TM TurboRVB
Eb_TM TurboRVB_DMCv2
Eb_TM QMCPACK_final
Eb_TM CMQMC
Eb_DLA CASINO
Eb_DLA TurboRVB
Eb_DLA QMCPACK_final
Eb_DLA CMQMC
Eb_DLTM CASINO
Eb_DLTM TurboRVB
Eb_DLTM CMQMC


In [11]:
code_final_list

['CASINO',
 'TurboRVB',
 'TurboRVB_DMCv2',
 'QMCPACK_final',
 'CMQMC',
 'CHAMP_Holland',
 'CHAMP_Cornell_Paris',
 'QMC=Chem',
 'QMeCha',
 'PyQMC',
 'Amolqc',
 'QWalk']

In [45]:
m

array([-24.8667839 , -25.21237375, -25.55563061, -25.89655448,
       -26.23514536, -26.57140325, -26.90532815, -27.23692005,
       -27.56617897, -27.8931049 , -28.21769784, -28.53995779,
       -28.85988474, -29.17747871, -29.49273969, -29.80566767,
       -30.11626267, -30.42452467, -30.73045369, -31.03404971,
       -31.33531275, -31.63424279, -31.93083984, -32.22510391,
       -32.51703498, -32.80663306, -33.09389816, -33.37883026,
       -33.66142937, -33.94169549, -34.21962862, -34.49522876,
       -34.76849591, -35.03943007, -35.30803124, -35.57429942,
       -35.83823461, -36.09983681, -36.35910602, -36.61604224,
       -36.87064546, -37.1229157 , -37.37285295, -37.6204572 ,
       -37.86572847, -38.10866675, -38.34927203, -38.58754433,
       -38.82348363, -39.05708995, -39.28836327, -39.5173036 ,
       -39.74391095, -39.9681853 , -40.19012666, -40.40973504,
       -40.62701042, -40.84195281, -41.05456221, -41.26483862,
       -41.47278204, -41.67839247, -41.88166991, -42.08

In [46]:
funlinspace

array([-24.87372225, -25.21807523, -25.56013392, -25.8998983 ,
       -26.2373684 , -26.5725442 , -26.9054257 , -27.23601291,
       -27.56430582, -27.89030444, -28.21400876, -28.53541879,
       -28.85453452, -29.17135595, -29.4858831 , -29.79811594,
       -30.10805449, -30.41569875, -30.72104871, -31.02410437,
       -31.32486574, -31.62333281, -31.91950559, -32.21338408,
       -32.50496826, -32.79425816, -33.08125375, -33.36595506,
       -33.64836206, -33.92847477, -34.20629319, -34.48181731,
       -34.75504714, -35.02598267, -35.2946239 , -35.56097084,
       -35.82502349, -36.08678184, -36.34624589, -36.60341565,
       -36.85829111, -37.11087228, -37.36115915, -37.60915173,
       -37.85485001, -38.098254  , -38.33936369, -38.57817909,
       -38.81470019, -39.048927  , -39.28085951, -39.51049772,
       -39.73784164, -39.96289127, -40.1856466 , -40.40610763,
       -40.62427437, -40.84014681, -41.05372496, -41.26500881,
       -41.47399837, -41.68069363, -41.8850946 , -42.08

In [44]:
xlinspace

array([0.   , 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008,
       0.009, 0.01 , 0.011, 0.012, 0.013, 0.014, 0.015, 0.016, 0.017,
       0.018, 0.019, 0.02 , 0.021, 0.022, 0.023, 0.024, 0.025, 0.026,
       0.027, 0.028, 0.029, 0.03 , 0.031, 0.032, 0.033, 0.034, 0.035,
       0.036, 0.037, 0.038, 0.039, 0.04 , 0.041, 0.042, 0.043, 0.044,
       0.045, 0.046, 0.047, 0.048, 0.049, 0.05 , 0.051, 0.052, 0.053,
       0.054, 0.055, 0.056, 0.057, 0.058, 0.059, 0.06 , 0.061, 0.062,
       0.063, 0.064, 0.065, 0.066, 0.067, 0.068, 0.069, 0.07 , 0.071,
       0.072, 0.073, 0.074, 0.075, 0.076, 0.077, 0.078, 0.079, 0.08 ,
       0.081, 0.082, 0.083, 0.084, 0.085, 0.086, 0.087, 0.088, 0.089,
       0.09 , 0.091, 0.092, 0.093, 0.094, 0.095, 0.096, 0.097, 0.098,
       0.099, 0.1  ])

In [43]:
xfit

array([0.   , 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008,
       0.009, 0.01 , 0.011, 0.012, 0.013, 0.014, 0.015, 0.016, 0.017,
       0.018, 0.019, 0.02 , 0.021, 0.022, 0.023, 0.024, 0.025, 0.026,
       0.027, 0.028, 0.029, 0.03 , 0.031, 0.032, 0.033, 0.034, 0.035,
       0.036, 0.037, 0.038, 0.039, 0.04 , 0.041, 0.042, 0.043, 0.044,
       0.045, 0.046, 0.047, 0.048, 0.049, 0.05 , 0.051, 0.052, 0.053,
       0.054, 0.055, 0.056, 0.057, 0.058, 0.059, 0.06 , 0.061, 0.062,
       0.063, 0.064, 0.065, 0.066, 0.067, 0.068, 0.069, 0.07 , 0.071,
       0.072, 0.073, 0.074, 0.075, 0.076, 0.077, 0.078, 0.079, 0.08 ,
       0.081, 0.082, 0.083, 0.084, 0.085, 0.086, 0.087, 0.088, 0.089,
       0.09 , 0.091, 0.092, 0.093, 0.094, 0.095, 0.096, 0.097, 0.098,
       0.099, 0.1  ])

In [38]:
xfit

array([0.        , 0.0010101 , 0.0020202 , 0.0030303 , 0.0040404 ,
       0.00505051, 0.00606061, 0.00707071, 0.00808081, 0.00909091,
       0.01010101, 0.01111111, 0.01212121, 0.01313131, 0.01414141,
       0.01515152, 0.01616162, 0.01717172, 0.01818182, 0.01919192,
       0.02020202, 0.02121212, 0.02222222, 0.02323232, 0.02424242,
       0.02525253, 0.02626263, 0.02727273, 0.02828283, 0.02929293,
       0.03030303, 0.03131313, 0.03232323, 0.03333333, 0.03434343,
       0.03535354, 0.03636364, 0.03737374, 0.03838384, 0.03939394,
       0.04040404, 0.04141414, 0.04242424, 0.04343434, 0.04444444,
       0.04545455, 0.04646465, 0.04747475, 0.04848485, 0.04949495,
       0.05050505, 0.05151515, 0.05252525, 0.05353535, 0.05454545,
       0.05555556, 0.05656566, 0.05757576, 0.05858586, 0.05959596,
       0.06060606, 0.06161616, 0.06262626, 0.06363636, 0.06464646,
       0.06565657, 0.06666667, 0.06767677, 0.06868687, 0.06969697,
       0.07070707, 0.07171717, 0.07272727, 0.07373737, 0.07474

In [ ]:
datac1 = data[code][f'{ed}_{method}']
datac1['x'] = x_axis( datac1, code )
datac = datac1[ datac1.x <= taumax ]
datac = datac.dropna()
x = datac.x
y = datac.ene
y_err = datac.err

In [30]:
# Fit the binding energy to an appropriate functional form

df_Eb



In [34]:
print(y_err)

tau
0.16    1.018064
0.08    1.154592
0.04    1.156725
0.02    1.087415
0.01    1.104371
Name: err, dtype: float64


In [25]:
Eb.values

array([-357.08430569, -135.33431539,  -62.53894937,  -35.46996488,
        -27.32075896,  -25.79808674,  -24.44703384])

In [28]:
Eb.x

AttributeError: 'Series' object has no attribute 'x'

In [23]:
dir(Eb)

['T',
 '_AXIS_LEN',
 '_AXIS_ORDERS',
 '_AXIS_REVERSED',
 '_AXIS_TO_AXIS_NUMBER',
 '_HANDLED_TYPES',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_ufunc__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__finalize__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__imod__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__long__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__r